#### Import Dependencies

In [2]:
import openai
import instructor
from qdrant_client import QdrantClient 

from pydantic import BaseModel, Field


#### RAG Pipeline

In [3]:
client = instructor.from_openai(openai.OpenAI())

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
class RAGGenerationResponse(BaseModel):
    answer: str=Field(description='The answer to the question')

In [7]:

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding
#### Retrieval function

def retrieve_similar_books(query, qdrant_client, top_k=5):

    query_embedding = get_embedding(query)
    
    search_result = qdrant_client.query_points(
        collection_name="amazon_books_collection_sampled",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in search_result.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context.append(result.payload['description'])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload['average_rating'])
    
    return {
        "context_ids": retrieved_context_ids,
        "context": retrieved_context,
        "similarity_scores": similarity_scores,
        "ratings": retrieved_context_ratings
    }
    

def process_context(retrieved_context):
    formatted_context = ""

    for id, chunk, rating in zip(retrieved_context["context_ids"], retrieved_context["context"], retrieved_context["ratings"]):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n\n"
    return formatted_context
#### Create prompt function

def build_prompt(preprocessed_context, user_query):
    prompt = f"""
    You are a book recommendation assistant that can answer questions about the books in stock.

    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products
    
    Context:
    {preprocessed_context}
    
    Question: {user_query}
    
    Answer:
    """
    return prompt


def generate_answer(prompt):
    response,raw_response = client.chat.completions.create_with_completion(
        model = 'gpt-4.1-mini',
        messages =[{"role": "system", "content": prompt}],
        temperature = 0,
        response_model=RAGGenerationResponse
    )
    return response


def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url= "http://localhost:6333")

    retrieved_context = retrieve_similar_books(question, qdrant_client, top_k)
    processed_context = process_context(retrieved_context)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        'answer': answer.answer,
        'question': question,
        'retrieved_context': retrieved_context,
        'retrieved_context_ids': retrieved_context["context_ids"],
        'similarity_scores': retrieved_context["similarity_scores"]
    }
    return final_result

In [8]:
output = rag_pipeline('Good historical fiction about civil war')

In [9]:
output

{'answer': 'A highly recommended historical fiction about the Civil War is "I Shall Be Near to You: A Novel." It tells the story of Rosetta, a strong-willed woman who disguises herself as a man to fight alongside her husband during the Civil War. The novel is inspired by the letters of a remarkable female soldier and is praised for its authentic and emotionally charged portrayal of war and enduring love. It offers a unique perspective on the Civil War and is well-regarded for its rich historical details and compelling narrative.',
 'question': 'Good historical fiction about civil war',
 'retrieved_context': {'context_ids': ['B00EBRUAVC',
   '1561640700',
   '161188263X',
   '006286808X',
   'B01DWTHKFI'],
  'context': ["I Shall Be Near to You: A NovelAn extraordinary novel about a strong-willed woman who disguises herself as a man in order to fight beside her husband, inspired by the letters of a remarkable female soldier who fought in the Civil War. Rosetta doesn't want her new husban

#### RAG Pipeline with Grounding Context

In [10]:
print(output['answer'])

A highly recommended historical fiction about the Civil War is "I Shall Be Near to You: A Novel." It tells the story of Rosetta, a strong-willed woman who disguises herself as a man to fight alongside her husband during the Civil War. The novel is inspired by the letters of a remarkable female soldier and is praised for its authentic and emotionally charged portrayal of war and enduring love. It offers a unique perspective on the Civil War and is well-regarded for its rich historical details and compelling narrative.


In [11]:
class RAGUsedContext(BaseModel):
    id : str = Field(description='The ID of the item used to answer the question')
    description: str = Field(description='short description of the item used to answer the question')

class RAGGenerationResponse(BaseModel):
    answer: str=Field(description='The answer to the question')
    references: list[RAGUsedContext]=Field(description='List of items used to the question')

In [18]:

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding
#### Retrieval function

def retrieve_similar_books(query, qdrant_client, top_k=5):

    query_embedding = get_embedding(query)
    
    search_result = qdrant_client.query_points(
        collection_name="amazon_books_collection_sampled",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in search_result.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context.append(result.payload['description'])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload['average_rating'])
    
    return {
        "context_ids": retrieved_context_ids,
        "context": retrieved_context,
        "similarity_scores": similarity_scores,
        "ratings": retrieved_context_ratings
    }
    

def process_context(retrieved_context):
    formatted_context = ""

    for id, chunk, rating in zip(retrieved_context["context_ids"], retrieved_context["context"], retrieved_context["ratings"]):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n\n"
    return formatted_context
#### Create prompt function

def build_prompt(preprocessed_context, user_query):
    prompt = f"""
    You are a book recommendation assistant that can answer questions about the books in stock.

    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products
    - You may recommend more than one book if you think it is contexually appropriate
    - As an output you need to provide

    * The answer to teh question based on the provided context
    * The list of IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
    * Short description (1-2 sentences) of the item based on the description provided in the context

    -The short description should have the name of the item.
    -The answer to the question should contain detailed information about the product and return with detailed specification in bullet points.

    Context:
    {preprocessed_context}
    
    Question: {user_query}
    
    Answer:
    """
    return prompt


def generate_answer(prompt):
    response,raw_response = client.chat.completions.create_with_completion(
        model = 'gpt-4.1-mini',
        messages =[{"role": "system", "content": prompt}],
        temperature = 0,
        response_model=RAGGenerationResponse
    )
    return response


def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url= "http://localhost:6333")

    retrieved_context = retrieve_similar_books(question, qdrant_client, top_k)
    processed_context = process_context(retrieved_context)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        'original_output': answer,
        'answer': answer.answer,
        'reference':answer.references,
        'question': question,
        'retrieved_context': retrieved_context,
        'retrieved_context_ids': retrieved_context["context_ids"],
        'similarity_scores': retrieved_context["similarity_scores"]
    }
    return final_result

In [22]:
result = rag_pipeline('I want to read about romance fiction.', top_k=10)

In [23]:
result

{'original_output': RAGGenerationResponse(answer="If you want to read romance fiction, there are several excellent options available:\n\n1. The Mammoth Book of Paranormal Romance (ID: 0762436514)\n   - Contains 24 new short stories from popular authors.\n   - Features paranormal romance with mythical beasts, magical creatures, ghosts, angels, and mortals with extra-sensitive sensory perception.\n   - Explores themes of extraordinary desires and love beyond boundaries.\n\n2. Romancing Miss Bronte: A Novel (ID: B008SLE2F2)\n   - A historical romance novel about Charlotte Brontë.\n   - Blends fact and fiction to portray Charlotte's life, literary success, and unexpected love.\n   - Set in the Yorkshire moors and London literary scene.\n\n3. Negotiation: A Supernatural Mystery and Romance (Masters of the Savoy Book 2) (ID: B097YQJQ12)\n   - A supernatural mystery romance involving a DSI and a dishonored history professor.\n   - Set in London with elements of paranormal and historical intri

In [24]:
print(result['answer'])

If you want to read romance fiction, there are several excellent options available:

1. The Mammoth Book of Paranormal Romance (ID: 0762436514)
   - Contains 24 new short stories from popular authors.
   - Features paranormal romance with mythical beasts, magical creatures, ghosts, angels, and mortals with extra-sensitive sensory perception.
   - Explores themes of extraordinary desires and love beyond boundaries.

2. Romancing Miss Bronte: A Novel (ID: B008SLE2F2)
   - A historical romance novel about Charlotte Brontë.
   - Blends fact and fiction to portray Charlotte's life, literary success, and unexpected love.
   - Set in the Yorkshire moors and London literary scene.

3. Negotiation: A Supernatural Mystery and Romance (Masters of the Savoy Book 2) (ID: B097YQJQ12)
   - A supernatural mystery romance involving a DSI and a dishonored history professor.
   - Set in London with elements of paranormal and historical intrigue.
   - Contains mature themes and steamy romance.

4. Can't H